# Parameter Reference — VaR Pipeline

Bu notebook VaR hattının uçtan uca akışını gösterir ve **değiştirilebilen her parametreyi** ilgili hücrenin başında `# >>> TUNABLE` bloğunda toplar; her parametrenin anlamı ve sınırları yanında yazılıdır. Bilgiler `data_management.py`, `feature_builders.py`, `tree_models.py` docstring ve kodlarından çıkarıldı.

**Akış:** `GroundTruthCalendar` → `build_*_features` → `*_train` (model + `ResidualBank` + `EwmaVarianceBank`) → `multi_day_var_from_*` → `calendar.backtest`

| # | Aşama | Fonksiyon | Tunable'lar |
|---|---|---|---|
| 1 | Veri / portföy | – | `TICKERS`, `WEIGHTS`, csv |
| 2 | Takvim | `DailyGroundTruthCalendar` | `train_*`, `forecast_*`, `drop_market_closed` |
| 3 | Feature | `build_long_features` / `build_wide_features` | `windows`, `ewma_span`, `sq_return_lags`, `target_scale` |
| 4 | Eğitim | `xgboost_train` / `random_forest_train` / `tree_train` | `rolling_mu_hat_window_size`, `mu_mode`, `default_mu_hat`, model hiperparametreleri |
| 5 | Long seed (gizli) | `long_format_train_seed` | `ewma_lam`, `ewma_min_periods` |
| 6 | Tahmin | `multi_day_var_from_*` | `multi_day_var_calculation_window_size`, `var_confidence`, `run_key` |
| 7 | Backtest | `calendar.backtest` | `var_confidence`, `test_confidence` |

> **En kritik ayrım:** `rolling_mu_hat_window_size` (drift/`mu_hat` penceresi, eğitimde `ResidualBank`'e girer) ile `multi_day_var_calculation_window_size` (VaR quantile penceresi, tahmin aşamasında) **birbirinden bağımsızdır**; aynı değere sahip olmaları gerekmez. Bu notebook'ta iki ayrı değişkenle verilirler.

In [ ]:
# Notebook alt klasörde duruyor: proje köküne geç (modüller ve ./data_general oradan okunur)
import os, sys
while not os.path.exists("data_management.py") and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import data_management as dm
from core import feature_builders as fb
import tree_models as tm
from graph_helpers import plot_var_vs_loss, plot_var_with_window_size

## 1. Veri ve portföy

| Parametre | Anlam | Not |
|---|---|---|
| `TICKERS` | Portföydeki hisseler | csv'de bu isimde kolon olmalı, yoksa `ValueError` |
| `WEIGHTS` | Ağırlıklar (dict veya Series) | Otomatik olarak toplamı 1'e normalize edilir; eksik ticker / sonsuz değer / toplam 0 → hata |
| `DATA_PATH` | **Ham fiyat** csv'si (`Date` index) | Takvim `pct_change()`'i kendisi alır. Ona **getiri değil fiyat** verilmeli (yoksa getirinin getirisi alınır) |

In [ ]:
# >>> TUNABLE
TICKERS = ["AAPL", "JPM", "XOM", "KO"]
WEIGHTS = {"AAPL": 0.25, "JPM": 0.25, "XOM": 0.25, "KO": 0.25}
DATA_PATH = "./data_general/4_ticker_data.csv"
# <<<

df = pd.read_csv(DATA_PATH, parse_dates=["Date"], index_col="Date")   # ham fiyatlar
df.tail(3)

## 2. Takvim (`GroundTruthCalendar`)

Sınıf frekansı belirler: `DailyGroundTruthCalendar` (24h), `HourlyGroundTruthCalendar` (1h), `MinuteGroundTruthCalendar` (1m). Hepsi aynı constructor imzasını kullanır.

| Parametre | Anlam | Kurallar |
|---|---|---|
| `train_start`, `train_end`, `forecast_start`, `forecast_end` | Eğitim / tahmin pencereleri | **Dördü de veride gerçek birer bar olmalı** (`KeyError`). `train_start <= train_end < forecast_start <= forecast_end` (look-ahead yasağı). Beklenen gün: start=Pazartesi, end=Cuma; değilse sadece `[WARNING]` basar |
| `drop_market_closed` | Veride olmayan hafta içi günler (tatil) | `True` → uyarı basıp atlar; `False` → `ValueError` fırlatır (daily frekansta) |

Buradan türeyenler: `calendar.train_data`, `calendar.forecast_data`, `calendar.forecast_dates`, `calendar.weights`, `calendar.tickers`, `calendar.frequency`.

Pencere uzunluğu doğrudan **kaç residual biriktiğini** belirler → `multi_day_var_calculation_window_size`'ın üst sınırı buradan gelir (bkz. Bölüm 5).

In [ ]:
# >>> TUNABLE
TRAIN_START    = "2024-06-03"   # Pazartesi
TRAIN_END      = "2024-11-29"   # Cuma
FORECAST_START = "2024-12-02"   # Pazartesi
FORECAST_END   = "2024-12-31"
DROP_MARKET_CLOSED = True
# <<<

calendar = dm.DailyGroundTruthCalendar(
    df, TICKERS, WEIGHTS,
    train_start=TRAIN_START, train_end=TRAIN_END,
    forecast_start=FORECAST_START, forecast_end=FORECAST_END,
    drop_market_closed=DROP_MARKET_CLOSED,
)

## 3. Feature builder

`build_long_features` ve `build_wide_features` **aynı 4 parametreyi** alır ve `(TrainingFeature, ForecastFeature)` döndürür.

| Parametre | Default | Anlam |
|---|---|---|
| `windows` | `(5, 10, 21)` | Her `w` için `real_vol_w` (rolling std), `mean_abs_w`, `skew_w` üretilir. **En büyük pencere kadar başlangıç barı warmup'ta düşer** (train başı kısalır) |
| `ewma_span` | `21` | `*_ewma_vol` ve (long'da) `market_ewma_vol` için EWMA span'i. Bu da warmup'a katkı yapar |
| `sq_return_lags` | `None` | Örn. `(1, 2, 3)` → `r2_lag_k` (gecikmeli kare getiri) feature'ları eklenir. Lag kadar daha satır düşer |
| `target_scale` | `1.0` | `y = (r_next)² * target_scale`. Kare getiriler çok küçük olduğu için ölçeklemek için artırılır. `TrainingFeature.target_scale` olarak saklanır; LSTM tahmininde `/ target_scale` ile geri çevrilir |

**Long vs Wide**
- **long:** (bar, ticker) başına bir satır, tüm tickerlar havuzda tek modelle eğitilir; hedef = o ticker'ın ertesi bar kare getirisi. Portföy varyansı `EwmaVarianceBank` ile kurulur (`wᵀ Σ w`).
- **wide:** tek satır/bar; feature'lar her ticker + `portfolio_return` için; hedef = portföyün ertesi bar kare getirisi.

> Feature'lar **takvime bağlı** üretilir. Takvim değişirse feature'lar da **yeniden üretilmelidir**; bir takvimden üretilen feature'lar başka bir takvimle kullanılmamalıdır.

In [ ]:
# >>> TUNABLE
WINDOWS        = (5, 10, 21)
EWMA_SPAN      = 21
SQ_RETURN_LAGS = None     # örn. (1, 2, 3)
TARGET_SCALE   = 1.0
# <<<

feat_kwargs = dict(windows=WINDOWS, ewma_span=EWMA_SPAN, sq_return_lags=SQ_RETURN_LAGS, target_scale=TARGET_SCALE)

training_feature_long, forecast_feature_long = fb.build_long_features(calendar, **feat_kwargs)
training_feature_wide, forecast_feature_wide = fb.build_wide_features(calendar, **feat_kwargs)

### Feature objelerinin içi (sadece inceleme, parametre yok)

| Alan | Anlam |
|---|---|
| `X_in_sample`, `y_in_sample` | Hedefi bilinen eğitim satırları |
| `X_out_sample_last_index` | Train penceresinin son satırı (hedefi henüz yok) → ilk forecast barının "pending" tahmini için |
| `X_in_sample_id_cols`, `X_out_sample_last_index_id_cols` | Sadece long: satır başına `Date`, `ticker` |
| `feature_dates` | Feature'ların gerçekten kapsadığı tarihler (warmup yüzünden `train_start`'tan geç başlayabilir) |
| `DANGEROUS_r_next_*` | Ertesi bar gerçek getiri — **sadece residual/backtest için**, modele feature olarak verilmemelidir |

In [ ]:
training_feature_long.X_in_sample_id_cols.tail(3)

In [ ]:
training_feature_long.X_in_sample.tail(3)

In [ ]:
training_feature_long.X_out_sample_last_index_id_cols

In [ ]:
training_feature_wide.X_in_sample.tail(3)

## 4. Model eğitimi

Üç giriş noktası: `xgboost_train`, `random_forest_train` (ikisi de `tree_train`'e sarmalayıcı) ve doğrudan `tree_train(calendar, training_feature, model_constructor, model_name, ...)` (başka bir sklearn tarzı regressor takmak için).

**Ortak (konumsal) imza:** `(calendar, training_feature, rolling_mu_hat_window_size, residual_bank_setting_mu_mode="constant", residual_bank_setting_default_mu_hat=0.0, **model_kwargs)`

`training_feature.type` `"long"` ise long yol, değilse wide yol otomatik seçilir.

> ⚠️ **Bilinen sınırlamalar (`tree_models.py`'nin mevcut hali):**
> 1. **Wide yol çalışmıyor:** `_tree_train_wide` içinde `ResidualBank(...)` pencereyi constructor'a vermiyor ve `bank.training_safe_bulk_insert(..., rolling_mu_hat_window_size=...)` çağrısı `training_safe_bulk_insert`'in kabul etmediği bir kwarg geçiyor → `TypeError`.
> 2. **`mu_mode="rolling"` long yolda çalışmıyor:** `_tree_train_long` önce pencere vermeden geçici bir `ResidualBank` kuruyor → `ValueError: mu_mode='rolling' requires a positive rolling_mu_hat_window_size, got: None`.
>
> Şu an çalışan kombinasyon: **long + `mu_mode="constant"`**.

| Parametre | Default | Anlam |
|---|---|---|
| `rolling_mu_hat_window_size` | **zorunlu** | `ResidualBank`'ta `mu_hat` (drift) hesabında kullanılan son-N-bar penceresi. **Sadece `mu_mode="rolling"` iken** sayıya etki eder; `"constant"`'ta okunmaz (ama yine de pozitif int verilmeli, imzada zorunlu) |
| `residual_bank_setting_mu_mode` | `"constant"` | `"constant"`: her bar için `mu_hat = default_mu_hat`. `"rolling"`: `mu_hat_t` = t'den **önceki** son N barın ortalama getirisi (t hariç, look-ahead yok) |
| `residual_bank_setting_default_mu_hat` | `0.0` | `"constant"`'ta sabit drift; `"rolling"`'de seed başlangıç değeri |
| `**model_kwargs` | aşağıda | Doğrudan `XGBRegressor` / `RandomForestRegressor`'a gider; defaults'u **ezer** |

**Model default'ları**

| Model | Default |
|---|---|
| xgboost | `n_estimators=100, max_depth=3, learning_rate=0.05, verbosity=0, n_jobs=1` |
| random forest | `n_estimators=200, max_depth=5, random_state=0` |

`standardized_residual = (actual_return - mu_hat) / sqrt(predicted_variance)`; VaR bu residual'ların quantile'ından türetilir.

In [ ]:
# >>> TUNABLE
ROLLING_MU_HAT_WINDOW_SIZE = 20         # sadece mu_mode="rolling" iken anlamlı
MU_MODE        = "constant"             # "constant" | "rolling"  (rolling: bkz. yukarıdaki bilinen sorun notu)
DEFAULT_MU_HAT = 0.0

XGB_KWARGS = dict(n_estimators=100, max_depth=3, learning_rate=0.05)   # varsayılanları ezer
RF_KWARGS  = dict(n_estimators=200, max_depth=5, random_state=0)
# <<<

train_kwargs = dict(
    residual_bank_setting_mu_mode=MU_MODE,
    residual_bank_setting_default_mu_hat=DEFAULT_MU_HAT,
)

model_xgb_long = tm.xgboost_train(calendar, training_feature_long, ROLLING_MU_HAT_WINDOW_SIZE, **train_kwargs, **XGB_KWARGS)
model_rf_long  = tm.random_forest_train(calendar, training_feature_long, ROLLING_MU_HAT_WINDOW_SIZE, **train_kwargs, **RF_KWARGS)

In [ ]:
# eğitim sonucu: model.residual_bank_in_sample  (ResidualBank)
#                model.ewma_variance_bank_in_sample (sadece long; EwmaVarianceBank)
model_xgb_long.residual_bank_in_sample.to_dataframe().tail()

In [ ]:
model_xgb_long.ewma_variance_bank_in_sample.to_dataframe().tail()

### 4b. Eğitimde dışarıdan ayarlanamayan ama ayarlanabilir olanlar (long)

`tree_train` → `_tree_train_long` → `long_format_train_seed(...)` zincirinde şu iki parametre **şu an `tree_train`'den geçirilmiyor**, `long_format_train_seed` default'larında sabit kalıyor:

| Parametre | Default | Anlam |
|---|---|---|
| `ewma_lam` | `0.94` | RiskMetrics EWMA decay'i (korelasyon/kovaryans için). Büyük → daha yavaş unutur |
| `ewma_min_periods` | `30` | EWMA burn-in: ilk N train barının örnek kovaryansı seed olur. `train_data`'da en az bu kadar bar gerekir. `usable_train_dates` eşiği `ewma_min_periods - 1`'dir; bu kadar bar bank'a girmez |

Değiştirmek için `long_format_train_seed` doğrudan çağrılır ve çıkan bank'lar modele elle atanır:

In [ ]:
# >>> TUNABLE (sadece long yol)
EWMA_LAM         = 0.94
EWMA_MIN_PERIODS = 30
# <<<

import copy
model_xgb_long_custom_ewma = copy.copy(model_xgb_long)   # sığ kopya yeterli: iki bank alanı yeniden atanıyor
(model_xgb_long_custom_ewma.residual_bank_in_sample,
 model_xgb_long_custom_ewma.ewma_variance_bank_in_sample) = tm.long_format_train_seed(
    model_xgb_long_custom_ewma, calendar, training_feature_long, ROLLING_MU_HAT_WINDOW_SIZE,
    ewma_lam=EWMA_LAM, ewma_min_periods=EWMA_MIN_PERIODS,
    residual_bank_setting_mu_mode=MU_MODE, residual_bank_setting_default_mu_hat=DEFAULT_MU_HAT,
)

## 5. Tahmin: VaR üretimi

İki fonksiyon, aynı mantık; hangisi kullanılacağı feature tipine bağlı:

| Feature tipi | Fonksiyon | Dönüş |
|---|---|---|
| long | `multi_day_var_from_long_format_model(model, calendar, forecast_feature, window, conf, residual_bank, ewma_bank, run_key=None)` | `(var_df, residual_bank, ewma_bank)` |
| wide | `multi_day_var_from_pretrained_model(model, forecast_feature, calendar, window, conf, run_key=None)` | `var_df` (**tek obje**, argüman sırası da farklı) |

| Parametre | Anlam |
|---|---|
| `multi_day_var_calculation_window_size` | **`int`: rolling pencere. `None`: expanding pencere (eldeki tüm residual'lar).** VaR quantile penceresi: t günü için t'den **önceki** son N standardized residual'ın `(1 - var_confidence)` quantile'ı alınır. **Mutlaka `ResidualBank`'taki geçmişle uyumlu olmalı**: bank'ta (train outcome + forecast outcome) kadar residual var, yetersizse `ValueError("Not enough prior residual history ...")`. Küçük N → hızlı adapte ama gürültülü; büyük N → stabil ama yavaş |
| `multi_day_var_min_periods` | Bir günün VaR alabilmesi için gereken en az önceki residual sayısı. Default: rolling'de `N` (pencere dolu olmalı), expanding'de `30`. Rolling'de `N`'den küçük verilirse kısmi pencere kabul edilir |
| `var_confidence` | Örn. `0.95` → %95 VaR, yani residual dağılımının alt %5 quantile'ı |
| `initial_residual_bank` / `initial_ewma_variance_bank` | Genelde `model.residual_bank_in_sample` / `model.ewma_variance_bank_in_sample`. Fonksiyon **kopyasını** kullanır, modelin kendi bank'ı değişmez |
| `run_key` | Sonucun `calendar.var_prediction_runs`'a yazıldığı anahtar. Default `model.model_name` → **aynı model türünü farklı ayarla tekrar çalıştırırsan üzerine yazar**, bu yüzden her deneye benzersiz anahtar ver |

**Çıktı `var_df`:** `varForecast`, `loss` ve `n_residuals_used` (o günün quantile'ının kaç residual ile hesaplandığı; rolling'de sabit `N`, expanding'de gün gün artar).

`calendar.forecast_pairs` her forecast gününü bir önceki barıyla eşler (ilk forecast günü için bu bar train'in son barıdır). Yani `n` forecast barından `n` VaR çıkar.

`rolling_mu_hat_window_size` bu aşamada **argüman değildir**; bank'a gömülü olarak gelir.

In [ ]:
# >>> TUNABLE
MULTI_DAY_VAR_WINDOW = 90     # quantile penceresi (bağımsız; ROLLING_MU_HAT_WINDOW_SIZE ile aynı olmak zorunda değil)
VAR_CONFIDENCE       = 0.95
# <<<

var_df, residual_bank, ewma_bank = dm.multi_day_var_from_long_format_model(
    model_xgb_long, calendar, forecast_feature_long,
    MULTI_DAY_VAR_WINDOW, VAR_CONFIDENCE,
    model_xgb_long.residual_bank_in_sample, model_xgb_long.ewma_variance_bank_in_sample,
    run_key="xgb_long",
)
var_df.head()

In [ ]:
plot_var_vs_loss(var_df, residual_bank)

### Random forest ile karşılaştırma (aynı pencere, ayrı `run_key`)

In [ ]:
var_df_rf, residual_bank_rf, _ = dm.multi_day_var_from_long_format_model(
    model_rf_long, calendar, forecast_feature_long,
    MULTI_DAY_VAR_WINDOW, VAR_CONFIDENCE,
    model_rf_long.residual_bank_in_sample, model_rf_long.ewma_variance_bank_in_sample,
    run_key="rf_long",
)
plot_var_vs_loss(var_df_rf, residual_bank_rf, title="RF: VaR Forecast vs Realized Loss")

### Expanding pencere: eldeki tüm veri

`multi_day_var_calculation_window_size=None` her gün için o günden önceki **bütün** residual'ları kullanır. `min_periods` (default 30) çok az veriyle quantile alınmasını engeller. Alt panelde her günün kaç residual ile hesaplandığı görünür.

In [ ]:
var_df_exp, residual_bank_exp, _ = dm.multi_day_var_from_long_format_model(
    model_xgb_long, calendar, forecast_feature_long,
    None, VAR_CONFIDENCE,                    # None = expanding
    model_xgb_long.residual_bank_in_sample, model_xgb_long.ewma_variance_bank_in_sample,
    run_key="xgb_long_expanding",
    multi_day_var_min_periods=30,
)
plot_var_with_window_size(var_df_exp, residual_bank_exp, title="Expanding window")

### Pencere duyarlılığı

Aynı eğitilmiş model, farklı `multi_day_var_calculation_window_size`. Eğitim yeniden yapılmaz (pencere sadece tahmin aşamasında devrede). Her deneyin `run_key`'i farklı.

In [ ]:
# >>> TUNABLE
WINDOW_GRID = [20, 60, 90]
# <<<

window_results = {}
for w in WINDOW_GRID:
    try:
        v, rb, _ = dm.multi_day_var_from_long_format_model(
            model_xgb_long, calendar, forecast_feature_long, w, VAR_CONFIDENCE,
            model_xgb_long.residual_bank_in_sample, model_xgb_long.ewma_variance_bank_in_sample,
            run_key=f"xgb_long_w{w}",
        )
        window_results[w] = v
    except ValueError as e:
        print(f"window={w}: {e}")

## 6. Wide akış

> ⚠️ Şu an `tree_models._tree_train_wide` kırık (Bölüm 4'teki not, madde 1) → aşağıdaki ilk hücre `TypeError` verir. Sınırlama giderildiğinde hücreler değişiklik gerektirmeden çalışacak şekilde yazılmıştır.

Aynı parametreler; farkları: `training_feature_wide` / `forecast_feature_wide` kullanılır, EWMA bank yoktur (`ewma_variance_bank_in_sample is None`), tahmin fonksiyonu `multi_day_var_from_pretrained_model`'dır.

In [ ]:
model_xgb_wide = tm.xgboost_train(calendar, training_feature_wide, ROLLING_MU_HAT_WINDOW_SIZE, **train_kwargs, **XGB_KWARGS)

var_df_wide = dm.multi_day_var_from_pretrained_model(
    model_xgb_wide, forecast_feature_wide, calendar,
    MULTI_DAY_VAR_WINDOW, VAR_CONFIDENCE, run_key="xgb_wide",
)
var_df_wide.head()

In [ ]:
plot_var_vs_loss(var_df_wide, model_xgb_wide.residual_bank_in_sample, title="Wide: VaR Forecast vs Realized Loss")

## 7. Backtest

`calendar.backtest(key, var_confidence, test_confidence=0.95)` — `key` bir `run_key` olmalı (`calendar.var_prediction_runs` anahtarları).

| Parametre | Anlam |
|---|---|
| `var_confidence` | Tahmin üretirken kullandığınla **aynı** olmalı (beklenen ihlal oranı `1 - var_confidence`) |
| `test_confidence` | İstatistiksel testlerin (Kupiec, Christoffersen vb.) güven seviyesi, default `0.95` |

In [ ]:
# >>> TUNABLE
TEST_CONFIDENCE = 0.95
BACKTEST_KEY    = "xgb_long"      # kayıtlı anahtarlar: list(calendar.var_prediction_runs)
# <<<

print(list(calendar.var_prediction_runs))
calendar.backtest(BACKTEST_KEY, VAR_CONFIDENCE, TEST_CONFIDENCE)

## Hızlı özet: hangi parametre nerede ve neyi etkiler

| Parametre | Nerede verilir | Etki alanı | Pozitif/geçerlilik şartı |
|---|---|---|---|
| `train_*`, `forecast_*` | Takvim | Her şey (veri miktarı, residual sayısı) | Gerçek bar, `train_end < forecast_start` |
| `drop_market_closed` | Takvim | Tatil boşluklarında uyarı vs. hata | – |
| `windows`, `ewma_span`, `sq_return_lags` | Feature builder | Feature seti + warmup kaybı | Pencere ≥ 2 (std/skew için) |
| `target_scale` | Feature builder | Hedef ölçeği (`y * scale`) | Pozitif |
| `model_kwargs` | `*_train` | Model kapasitesi | Modele bağlı |
| `rolling_mu_hat_window_size` | `*_train` → `ResidualBank` | `mu_hat` penceresi (yalnız `rolling`) | Pozitif int (zorunlu argüman) |
| `mu_mode`, `default_mu_hat` | `*_train` → `ResidualBank` | Residual standardizasyonu ve VaR'daki `-mu_hat` terimi | `"rolling"` / `"constant"` |
| `ewma_lam`, `ewma_min_periods` | `long_format_train_seed` (tree_train'den geçmiyor) | Long yolda portföy kovaryansı | `0<lam<1`, `train_data >= min_periods` bar |
| `multi_day_var_calculation_window_size` | Tahmin | VaR quantile penceresi (`None` = expanding) | Rolling'de bank geçmişinden büyük olamaz |
| `multi_day_var_min_periods` | Tahmin | Bir günün VaR alması için gereken en az residual | Pozitif int; default rolling'de `N`, expanding'de 30 |
| `var_confidence` | Tahmin + Backtest | Quantile seviyesi `1 - conf` | `0 < conf < 1` |
| `run_key` | Tahmin | `calendar.var_prediction_runs` kaydı | Tekrar kullanılırsa üzerine yazar |
| `test_confidence` | Backtest | İstatistiksel test güveni | `0 < tc < 1` |